# Getting Started with cande-wrapper

This notebook demonstrates how to use the `cande-wrapper` Python package to run CANDE
(Culvert ANalysis and DEsign) finite element analyses.

**Prerequisites:** The package must be installed with the compiled Fortran extension.
See `GETTING_STARTED.md` for installation instructions.

```bash
pip install -e ".[dev]"
```

## 1. Import the package

In [1]:
from cande_wrapper import CandeEngine
from pathlib import Path
import shutil
import tempfile

ModuleNotFoundError: No module named 'cande_wrapper'

## 2. Set up a working directory

CANDE reads input from `.cid` files and writes output files to the same directory.
We'll copy the included example input file into a temporary working directory.

In [ ]:
# Create a temporary working directory
work_dir = Path(tempfile.mkdtemp(prefix="cande_demo_"))
print(f"Working directory: {work_dir}")

# Copy the example input file
example_cid = Path("tests/example_data/MGK-IO.cid")
shutil.copy2(example_cid, work_dir / "MGK-IO.cid")

print(f"Input file: {work_dir / 'MGK-IO.cid'}")

## 3. Create the engine and verify input

The `CandeEngine` takes a working directory where `.cid` files live.

In [ ]:
engine = CandeEngine(work_dir=work_dir)

# Verify the input file exists
print(f"Input file exists: {engine.check_input('MGK-IO')}")

## 4. Inspect the input file

CANDE `.cid` files are fixed-format text. Let's look at the first few lines.

In [ ]:
cid_text = (work_dir / "MGK-IO.cid").read_text()
for i, line in enumerate(cid_text.splitlines()[:15], 1):
    print(f"{i:3d}: {line}")

## 5. Run the analysis

Call `engine.run()` with the file prefix (without `.cid` extension).
This runs the CANDE Fortran engine and returns a `CandeResult` object.

> **Note:** This cell requires the compiled Fortran extension.
> If you haven't built it yet, see `GETTING_STARTED.md`.

In [ ]:
result = engine.run("MGK-IO")
print(result)

## 6. Inspect the results

The `CandeResult` object provides paths to all output files.

In [ ]:
print(f"Output file: {result.output_file}")
print(f"Log file:    {result.log_file}")
print(f"TOC file:    {result.toc_file}")
print()

# List all files created in the working directory
print("All files in working directory:")
for f in sorted(work_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

## 7. Read the output report

The `.out` file contains the full CANDE analysis report.

In [ ]:
output = result.output_text
lines = output.splitlines()
print(f"Total output lines: {len(lines)}")
print()
print("=== First 50 lines ===")
for line in lines[:50]:
    print(line)

In [ ]:
# Check for normal completion
if "NORMAL EXIT FROM CANDE" in output:
    print("Analysis completed successfully.")
else:
    print("WARNING: Normal exit message not found in output.")

## 8. Clean up

In [ ]:
shutil.rmtree(work_dir)
print(f"Cleaned up {work_dir}")

## Next Steps

- Read `GETTING_STARTED.md` for full documentation
- Create your own `.cid` input files (see the CANDE User Manual PDF in `CANDE Source/`)
- Parse the XML output files for post-processing and visualization
- Run parametric studies by modifying `.cid` files programmatically